# L10c: Stochastic Multi-Armed Bandit Problems
In this lecture, we continue our exploration of Reinforcement Learning (RL), a branch of machine learning focused on how agents should take actions in an environment to maximize cumulative reward over time. We focus on multi-armed bandit problems, where the agent learns only from the reward of each action it takes.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Balance exploration and exploitation:__ An agent that learns from rewards must choose between trying actions to learn more about them (exploration) and taking the action that looks best so far (exploitation). We'll see why an agent needs both, and how each algorithm in this lecture balances them.
> * __Solve multi-armed bandit problems:__ In a multi-armed bandit problem, an agent repeatedly chooses one of several actions (arms), each with an unknown reward distribution, and tries to collect as much reward as possible. We'll work through the explore-first, epsilon-greedy, UCB1, and Thompson sampling algorithms for choosing the next arm.
> * __Compare bandit algorithms with regret:__ Regret is the reward an agent loses by not always choosing the best arm. We'll define the regret of a bandit algorithm and compare the expected-regret bounds for explore-first, epsilon-greedy, and UCB1.

The agent has to learn while it acts: every choice earns a reward and provides evidence about the action it tried. Let's get started!
___

## Exploration vs. Exploitation
Suppose an agent can choose among several actions, each with an unknown reward distribution. The agent chooses an action, receives a reward, and uses that observation to decide what to try next. Its goal is to maximize the reward collected over many rounds.

The agent must make decisions based on incomplete information and balance two competing objectives, exploration and exploitation:
1. **Exploration**: Trying new actions to discover potentially better rewards. This is essential for learning about the environment and finding the best actions. Taking purely random actions, or actions that have not been tried often, to gather information about their outcomes and rewards is an example of exploration.
2. **Exploitation**: Using current knowledge to choose the action that looks best right now. This involves choosing actions that have previously yielded high rewards based on the agent's experience. However, if the agent only exploits, it may miss out on discovering better actions that could yield higher rewards in the long run. The agent never tries anything new; how boring!

If an agent explores too much, it may miss out on immediate rewards. If it exploits too much, it may write off an action whose low observed reward was simply bad luck, especially if we have tried it only once. The algorithms in this lecture strike this balance in different ways: some explore on a fixed schedule, while others direct their exploration toward the actions whose rewards remain most uncertain.

__Discuss:__ Think of a choice you make again and again, such as where to eat lunch. How do you decide when to try something new and when to go back to your favorite?
___


## What Is a Bandit Problem?
A bandit problem is a class of online (sequential) decision-making tasks in which an agent repeatedly chooses among $K$ options (called _arms_) and receives a reward based on that choice.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-Bandit-Arms/Fig-Bandit-Arms.svg" width="640" style="max-width:100%; height:auto;" alt="Four vertical reward axes labeled arm 1 to arm 4, each running from reward 0 at the bottom to 1 at the top, with a shaded reward distribution drawn beside it and a horizontal tick at its mean reward, labeled mu of 1 to mu of 4. A dashed line across all four arms marks mu star, the mean reward of arm 3, the highest. A red bracket beside arm 1 spans the gap Delta of 1, from the mean reward of arm 1 up to mu star. Arm 2 is shaded blue and labeled pulled, a sub t equals 2, with a blue dot on its axis labeled r sub t, the one reward observed; the other three arms are labeled no reward observed."/>
</p>

__Only the pulled arm reports a reward.__ Each arm $a$ has a reward distribution $\mathcal{D}_a$ on $[0,1]$ (the shaded shape) with mean reward $\mu(a)$ (the horizontal tick), and the agent can see neither. In round $t$, the agent pulls arm $a_t$ and observes one reward $r_t$ drawn from $\mathcal{D}_{a_t}$; the other arms report nothing. The dashed line marks the largest mean reward, $\mu^\star$, and the red bracket marks the gap $\Delta(1)=\mu^\star-\mu(1)$.

In each application below, a system learns by trying one option at a time, sees the outcome of only that option, and pays a real cost when it explores:

| Application | Arms | Reward of one pull | Cost of exploring |
|:---|:---|:---|:---|
| Clinical trials ([Villar et al., 2015](https://pmc.ncbi.nlm.nih.gov/articles/PMC4856206/)) | treatments tested on volunteer patients | 1 if the treatment works for the patient, 0 if not | some patients get the less effective treatment |
| News recommendation ([Li et al., 2010](https://arxiv.org/abs/1003.0146)) | articles for the featured spot on the Yahoo! front page | 1 if the visitor clicks the article, 0 if not | some visitors see a less popular article |
| Network routing ([Talebi et al., 2018](https://arxiv.org/abs/1309.7367)) | paths through a computer network from a source to a destination | the time a chunk of data takes to arrive (a cost to minimize) | some data take a slower path |
| Online pricing ([Misra et al., 2019](https://doi.org/10.1287/mksc.2018.1129)) | candidate prices for a product | the profit on the sale, or 0 if the customer does not buy | some customers see a less profitable price |
| Reaction conditions ([Wang et al., 2024](https://doi.org/10.1038/s41586-024-07021-y)) | candidate combinations of catalyst, solvent, base, and other conditions | the yield of one test reaction, as a fraction between 0 and 1 | some experiments use reagents and time on poor conditions |

Delay and profit are not rewards between 0 and 1, so the regret bounds below apply only after a conversion: we rescale each one to lie between 0 and 1, with a faster arrival or a higher profit closer to 1.

Bandit methods trace back to [Thompson (1933)](https://doi.org/10.1093/biomet/25.3-4.285), who suggested dividing individuals between two treatments by giving each new individual the first treatment with probability equal to the chance, given the outcomes so far, that it is the better one. That rule, extended to more than two arms, is the Thompson sampling algorithm at the end of this lecture.

### Stochastic Bandit Model
In the **stationary stochastic bandit problem**, the agent has a fixed set of arms $\mathcal{A}$, with $|\mathcal{A}|=K$, and plays for $T$ rounds. The mean reward of arm $a$ is $\mu(a)=\mathbb{E}_{r\sim\mathcal{D}_a}[r]$. The regret bounds below rely on three assumptions: each pull is an independent draw, the distributions do not change (stationary), and every reward lies between 0 and 1.

For $t=1,2,\dots,T$ __do__:
1. The agent chooses an arm $a_t\in\mathcal{A}$ using its previous actions and observed rewards.
2. The environment independently draws $r_t\sim\mathcal{D}_{a_t}$.
3. The agent receives $r_t$ and updates its estimates using that observation.

The goal of the agent is to maximize its total reward over the $T$ rounds. Equivalently, the algorithm designer tries to minimize the _regret_ of the algorithm that the agent uses to choose its arms.

### Regret
Regret measures the difference between what could have been achieved by always making the best decision and what the agent actually received. Let $\mu^\star=\max_{a\in\mathcal{A}}\mu(a)$ be the highest **true mean reward** (the dashed line in the figure). Always pulling the best arm earns $T\mu^\star$ in expectation, so the regret of one game (one run of $T$ rounds) is given by:
$$
R(T)=T\mu^\star-\sum_{t=1}^{T}r_t.
$$
The first term is an expected reward, while the second term adds up the rewards actually received, so random reward fluctuations move $R(T)$ up and down, even after the agent has learned to pull the best arm.

When we know the arm means, we can also evaluate the **cumulative gap loss**, often called pseudo-regret. The gap of arm $a$ is $\Delta(a)=\mu^\star-\mu(a)$ (the red bracket in the figure), and the cumulative gap loss is given by:
$$
G(T)=\sum_{t=1}^{T}\Delta(a_t)=\sum_{t=1}^{T}\left(\mu^\star-\mu(a_t)\right).
$$
Each pull adds the mean reward given up by choosing that arm, so a pull of the best arm adds nothing. The gap loss still varies from game to game, because the arms an algorithm selects depend on the rewards it has observed. Since a pull of arm $a_t$ has expected reward $\mu(a_t)$, the two measures agree on average:
$$
\mathbb{E}[R(T)]=\mathbb{E}[G(T)].
$$
The bounds below concern this expected regret, averaged over the reward draws and the algorithm's own random choices.

__Additional Resources__: Our lecture notes for this week were inspired by Chapter 1 of [Introduction to Multi-Armed Bandits by Aleksandrs Slivkins](https://arxiv.org/abs/1904.07272), an excellent (albeit quite technical) resource for learning more about bandit problems. We also drew material from the [Thompson sampling tutorial by Russo et al., 2020](https://arxiv.org/abs/1707.02038).

___

## Bandit Algorithms
A bandit algorithm is the rule the agent uses to choose its next arm from its previous actions and observed rewards (step 1 of the stochastic bandit model). We consider three such rules.

### Explore-First Exploration
A straightforward approach to the multi-armed bandit problem is to explore each arm equally. This is called _uniform exploration_ or the **explore-first algorithm**: the agent pulls each arm $N$ times (the _exploration phase_), then pulls the arm with the highest sample mean reward for the rest of the game (the _exploitation phase_).

#### Explore-First Algorithm
__Initialize__: Given $K$ arms, $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the number of rounds $T\geq K$, choose the number of pulls per arm, an integer $N$ with $1\leq N\leq\lfloor T/K\rfloor$. We use:
$$
N=\min\left\{\left\lfloor\frac{T}{K}\right\rfloor,\;\max\left\{1,\;\operatorname{round}\left[\left(\frac{T}{K}\right)^{2/3}(\log T)^{1/3}\right]\right\}\right\}.
$$
Here, $\log$ is the natural logarithm. The $\max$ makes sure every arm is pulled at least once, and the $\min$ makes sure the exploration fits in $T$ rounds.

1. For each arm $a\in\mathcal{A}$ __do__:
   - Pull arm $a$ for $N$ rounds and record its rewards.
   - Compute its sample mean $\bar{\mu}(a)$, the sum of its $N$ rewards divided by $N$.
2. Choose the arm with the highest sample mean, $\hat{a}\gets\arg\max_{a\in\mathcal{A}}\bar{\mu}(a)$, breaking ties by the first arm index.
3. For $t=KN+1,KN+2,\dots,T$ __do__:
   - Pull arm $\hat{a}$ and receive a reward $r_{t}$ from the environment.

__Theorem__: Under the stated stochastic reward assumptions, the explore-first algorithm with $N$ of order $(T/K)^{2/3}(\log T)^{1/3}$ achieves the expected-regret bound $\mathbb{E}[R(T)]\leq T^{2/3}\cdot\mathcal{O}(K\log T)^{1/3}$ ([Slivkins, Theorem 1.5](https://arxiv.org/abs/1904.07272)). This choice of $N$ balances the cost of exploration against the cost of committing to a poorly estimated arm.

Here, "of order $x$" means a constant multiple of $x$, and the big-O notation $\mathcal{O}(x)$ means at most a constant multiple of $x$; neither constant depends on $T$ or $K$.


### Epsilon-Greedy Exploration
One issue with uniform exploration is that the exploration phase may be costly if many of the arms have a large gap $\Delta(a)$, because every exploration pull of such an arm gives up $\Delta(a)$ in expected reward. In that case, it may be better to spread out (and interleave) exploration and exploitation.

This is the idea behind the **epsilon-greedy algorithm**: the agent pulls the best arm so far with probability $1-\epsilon_t$ and a random arm with probability $\epsilon_t$, where the exploration probability $\epsilon_t$ shrinks as observations accumulate. In other sources, $\epsilon$ is a constant, not a function of $t$.

#### Epsilon-Greedy Algorithm
__Initialize__: Given $K$ arms, $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the number of rounds $T\geq K$, pull each arm $a\in\mathcal{A}$ once and denote its observed reward by $r_{a}$. For each arm $a\in\mathcal{A}$, set the pull count $N_{a}\gets1$ and the sample mean $\bar{\mu}(a)\gets r_{a}$.

For $t=K+1,K+2,\dots,T$ __do__:
1. Compute the exploration probability and draw $u$ uniformly from $[0,1]$. Our exploration probability is given by:
   $$
   \epsilon_t=\min\left\{1,\left(\frac{K\log t}{t}\right)^{1/3}\right\}.
   $$
2. Choose the arm $a_{t}$:
   - If $u\leq\epsilon_t$ __then__ choose $a_{t}\in\mathcal{A}$ uniformly at random (exploration).
   - Otherwise, choose the arm with the highest sample mean, $a_{t}\gets\arg\max_{a\in\mathcal{A}}\bar{\mu}(a)$, breaking ties by the first arm index (exploitation).
3. Pull arm $a_{t}$ and receive a reward $r_{t}$ from the environment.
4. Update the pull count $N_{a_{t}}\gets N_{a_{t}}+1$ and the sample mean $\bar{\mu}(a_{t})$, the sum of the arm's rewards divided by its pull count.

The cap keeps $\epsilon_t$ a valid probability in the early rounds, when $K\log t>t$.

__Theorem__: The epsilon-greedy algorithm above, with the capped exploration probability $\epsilon_t$, achieves the expected-regret bound $\mathbb{E}[R(t)]\leq t^{2/3}\cdot\mathcal{O}(K\log t)^{1/3}$ for each round $t$. [Slivkins (Theorem 1.6)](https://arxiv.org/abs/1904.07272) states this bound for $\epsilon_t=t^{-1/3}(K\log t)^{1/3}$ without the cap or the initial pulls, which do not change its order.


### Optimism Under Uncertainty
Let's consider an approach that adapts exploration to the observed rewards: __optimism under uncertainty__. Assume each arm is as good as it can be given the observations so far, and choose the best arm based on these optimistic estimates. This intuition leads to the UCB1 algorithm, proposed by [Auer, Cesa-Bianchi, and Fischer in 2002](https://doi.org/10.1023/A:1013689704352).

__Definition__: Upper Confidence Bound (UCB). The upper confidence bound of arm $a\in\mathcal{A}$ at round $t$ is the sum $\bar{\mu}(a)+U(a,t)$, where $\bar{\mu}(a)$ is the sample mean reward of arm $a$ and $U(a,t)$ is its _confidence radius_, an allowance for how uncertain that sample mean still is, given by:
$$
\begin{align*}
U(a,t) = \sqrt{\frac{2\log(t)}{N_{a}}},
\end{align*}
$$
where $N_{a}$ is the number of times that arm $a$ has been pulled before round $t$. Under the stated stochastic reward assumptions, the upper confidence bound lies above the true mean reward $\mu(a)$ with high probability.

#### UCB1 Algorithm
__Initialize__: Given $K$ arms, $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the number of rounds $T\geq K$, pull each arm $a\in\mathcal{A}$ once and denote its observed reward by $r_{a}$. For each arm $a\in\mathcal{A}$, set $N_{a}\gets1$ and $\bar{\mu}(a)\gets r_{a}$.

For rounds $t = K+1,K+2,\dots,T$ __do__:
1. Compute the confidence radius $U(a,t)$ for each arm $a\in\mathcal{A}$.
2. Choose the arm with the largest upper confidence bound, $a_{t}\gets\arg\max_{a\in\mathcal{A}}\left\{\bar{\mu}(a)+U(a,t)\right\}$, breaking ties by the first arm index.
3. Pull arm $a_{t}$ and receive a reward $r_{t}$ from the environment.
4. Update the pull count $N_{a_{t}}\gets N_{a_{t}}+1$ and the sample mean $\bar{\mu}(a_{t})$, the sum of the arm's rewards divided by its pull count.

__Theorem__: Under the stated stochastic reward assumptions, the UCB1 algorithm achieves the expected-regret bound $\mathbb{E}\left[R(T)\right]\leq\mathcal{O}\left(\sqrt{KT\cdot\log(T)}\right)$ over $T$ rounds ([Slivkins, Theorem 1.15](https://arxiv.org/abs/1904.07272)). Slivkins proves it with $\log T$ in the radius; the original $\log t$ radius gives the same bound.

#### Worked Example: An Uncertain Arm Receives Another Pull
Suppose two arms have been pulled for five rounds. Arm 1 has been pulled four times and returned rewards summing to 3, so $\bar{\mu}(1)=0.75$; arm 2 has been pulled once and returned zero, so $\bar{\mu}(2)=0$. Before round $t=6$, the upper confidence bound of arm 2 is given by:
$$
\bar{\mu}(2)+U(2,6)=0+\sqrt{\frac{2\log 6}{1}}\approx1.893,
$$
and the same calculation with $N_{1}=4$ gives $0.750+0.947=1.697$ for arm 1.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<p align="center">
    <img class="course-diagram" src="figs/Fig-UCB-Example/Fig-UCB-Example.svg" width="600" style="max-width:100%; height:auto;" alt="Two horizontal bars on a reward scale from 0 to 2. Arm 1, pulled 4 times, has an open dot at its sample mean 0.750 and a gray bar of length 0.947, its confidence radius, ending at its upper confidence bound 1.697. Arm 2, pulled once, has an open dot at its sample mean 0 and a longer blue bar of length 1.893 ending at its upper confidence bound 1.893, labeled pull arm 2. A dashed vertical line at 1 marks the largest possible reward; both bars cross it."/>
</p>

__One observation leaves more uncertainty than four.__ Each open dot is an arm's sample mean, and its bar extends by the confidence radius to the upper confidence bound. The algorithm selects arm 2, even though its sample mean is lower. The confidence radius is an allowance for uncertainty, so an upper confidence bound can exceed the largest possible reward of one (the dashed line).

If the next reward of arm 2 is 1, its pull count becomes 2, its reward sum becomes 1, and its sample mean becomes $1/2$.

### Comparing the Regret Bounds
The expected-regret bounds grow at different rates with the number of rounds $T$:

| Algorithm | How it explores | Expected-regret bound |
|---|---|---|
| Explore-first | pulls every arm $N$ times, then commits to one arm | $T^{2/3}\cdot\mathcal{O}(K\log T)^{1/3}$ |
| Epsilon-greedy | pulls a random arm with probability $\epsilon_t$, which shrinks over time | $T^{2/3}\cdot\mathcal{O}(K\log T)^{1/3}$ |
| UCB1 | pulls the arm with the largest upper confidence bound | $\mathcal{O}\left(\sqrt{KT\log T}\right)$ |

All three grow more slowly than $T$, so the expected regret per round shrinks toward zero as the game gets longer. The UCB1 bound grows with about $\sqrt{T}$ rather than $T^{2/3}$, so UCB1 has the strongest guarantee of the three.

__Discuss:__ UCB1 has the strongest guarantee, yet explore-first is still common in practice. When might it be the better choice, for example when each pull is a slow experiment that we must plan in advance?

___

## Binary Bernoulli Bandit Problem
The binary Bernoulli bandit problem is a special case of the stochastic bandit problem where the reward for taking action $a\in\mathcal{A}$ is binary, $r_{t} \in \left\{0,1\right\}$. For each arm $a$, let $\theta_a$ denote its unknown probability of receiving reward `1`, so its mean reward is $\mu(a)=\theta_a$.

* _Reward model_: The reward distribution of each arm has a known form, [the Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution), with an unknown success probability. This gives the agent a model of how the world responds to each action (which is so super cool!), and the observations help it estimate that probability.
* _Binary_: The reward distribution is binary, but the experiment represented by the action $a$ can be a complex statement or function that _evaluates_ to a boolean value. Thus, we can model many complex scenarios that evaluate to `true` or `false`.

The Bernoulli distribution returns `1` with probability $\theta$ and `0` with probability $1-\theta$, so its probability mass function is given by:
$$
\begin{equation*}
\texttt{Bern}(r; \theta) = \begin{cases}
\theta & \text{if } r = 1\\
1-\theta & \text{if } r = 0
\end{cases},
\end{equation*}
$$
where $r\in\left\{0,1\right\}$ is the reward. If $r\sim\texttt{Bern}(\theta)$, the expected reward is $\mathbb{E}[r] = \theta$ and its variance is $\text{Var}[r] = \theta(1-\theta)$. For arm $a$, we use this distribution with $\theta=\theta_a$.

_Ready to get your mind blown_? Ok, so here is the _cool part_: we can represent our uncertainty about each success probability $\theta_a$ by choosing a _prior probability distribution_ and updating this distribution as we observe rewards. This is the essence of the [Bayesian approach to bandit problems](https://onlinelibrary.wiley.com/doi/10.1002/asmb.874), and [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) is a natural choice for the prior:

> __Beta distribution:__ The distribution $\text{Beta}(\alpha,\beta)$ describes our uncertainty about an unknown probability $\theta\in[0,1]$. The parameters $\alpha>0$ and $\beta>0$ set its shape, and its mean is $\alpha/(\alpha+\beta)$. Suppose our belief about $\theta$ before we see any rewards (the __prior__) is $\text{Beta}(\alpha,\beta)$, where $\text{Beta}(1,1)$ is the uniform distribution on $[0,1]$. After we observe $S$ successes and $F$ failures, our updated belief (the __posterior__) is again a Beta distribution, $\text{Beta}(\alpha+S,\,\beta+F)$.

Yeah. That's cool. But how do we solve this problem?

### $\epsilon$-Greedy Binary Bernoulli Bandit
The epsilon-greedy algorithm above carries over to the binary Bernoulli bandit with two changes. First, the agent tracks the number of successes $S(a)$ and failures $F(a)$ of each arm in place of its pull count and sample mean, starting from $S(a)=F(a)=0$ and uniform priors, $\alpha(a)=\beta(a)=1$. Second, when it exploits, it chooses the action with the _highest expected probability of success_ under its model of the world, the arm with the highest posterior mean:
$$
a_{t}\gets\arg\max_{a\in\mathcal{A}}\left\{\frac{\alpha(a) + S(a)}{\alpha(a) + \beta(a) + S(a) + F(a)}\right\}.
$$
After each pull, the agent updates the counts of the arm it pulled using the reward $r_{t}\in\left\{0,1\right\}$:
$$
S(a_{t}) \gets S(a_{t}) + r_{t},\quad F(a_{t}) \gets F(a_{t}) + (1-r_{t}).
$$
Everything else is unchanged: the agent pulls each arm once, then explores with the capped probability $\epsilon_t$ by choosing an arm uniformly at random.

### Thompson Sampling Binary Bernoulli Bandit
An alternative Bayesian approach is Thompson sampling: in each round, the agent draws one sample from each arm's posterior distribution and pulls the arm with the highest sample.

#### Thompson Sampling Algorithm
__Initialize__: Given $K$ arms, $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the number of rounds $T\geq1$, set the Beta prior parameters $\alpha(a)\gets1$ and $\beta(a)\gets1$ for each arm $a\in\mathcal{A}$, along with the number of successes and failures $S(a)\gets0$ and $F(a)\gets0$.

For $t = 1,2,\dots,T$ __do__:
1. Sample from the posterior for each arm: $p(a)\sim\text{Beta}\left(\alpha(a)+S(a),\,\beta(a)+F(a)\right)$ for every $a\in\mathcal{A}$, where $S(a)$ and $F(a)$ are the number of successes and failures for arm $a$.
2. Choose the action with the highest sampled probability: $a_{t}\gets\arg\max_{a\in\mathcal{A}}\left\{p(a)\right\}$.
3. Execute the action $a_{t}$ and receive a reward $r_{t}\in\left\{0,1\right\}$ from the environment.
4. Update the number of successes $S(a_{t})$ and failures $F(a_{t})$ for the chosen arm $a_{t}$ using the reward $r_{t}$:
   $$
   S(a_{t}) \gets S(a_{t}) + r_{t},\quad F(a_{t}) \gets F(a_{t}) + (1-r_{t}).
   $$

Thompson sampling is fully Bayesian and often performs better than $\epsilon$-greedy in practice, because its exploration goes mostly to arms that could still be the best, rather than uniformly to every arm ([Russo et al., 2020](https://arxiv.org/abs/1707.02038)).

#### Worked Example: Updating a Bernoulli Arm
With uniform $\operatorname{Beta}(1,1)$ priors, suppose arm 1 has three successes and one failure, while arm 2 has no successes and one failure. Their posteriors are $\operatorname{Beta}(4,2)$ and $\operatorname{Beta}(1,2)$, with means $2/3$ and $1/3$.

Suppose the next posterior samples are 0.55 for arm 1 and 0.70 for arm 2. Thompson sampling chooses arm 2 because its sample is larger. If that pull succeeds, arm 2 now has one success and one failure, and its posterior becomes $\operatorname{Beta}(2,2)$ with mean $1/2$. Arm 1's posterior stays unchanged because we received no new observation from that arm. Sampling lets an uncertain arm receive another trial even when its posterior mean is lower.

__Discuss:__ Suppose each arm is a set of reaction conditions and each pull is one experiment, a success if the yield clears a target. After a few weeks of experiments, what do the posteriors tell us that the success counts alone do not?

___

## Lab
In the [lab](../L10d/CHEME-5800-L10d-Lab-BanditAlgorithms-Fall-2026.ipynb), we run the explore-first, epsilon-greedy, and UCB1 algorithms on a bandit with seven arms and compare their regret with the bounds above. We then implement Thompson sampling for a binary Bernoulli bandit and watch the posterior of the best arm concentrate around its true success probability, while the arms it rarely pulls stay uncertain.

___

## Summary
In this lecture, we introduced the stochastic multi-armed bandit problem and four algorithms for balancing exploration and exploitation.

> __Key Takeaways:__
>
> * __Exploration and exploitation:__ We saw that an agent learning from rewards must both try actions to learn about them and take the action that looks best so far. Too much exploration wastes rewards on poor arms, while too little can lock the agent onto an arm that only looked good by chance.
>
> * __Bandit algorithms:__ We worked through four ways to choose the next arm: explore every arm first and then commit, explore at random with a shrinking probability (epsilon-greedy), add an optimism bonus to each arm's estimated reward (UCB1), and sample from a Beta model of each arm's success probability (Thompson sampling). For binary rewards, the Beta model turns the successes and failures of each arm into a distribution for its success probability, which the agent updates after every pull.
>
> * __Regret bounds:__ We measured each algorithm by its regret, the reward lost by not always choosing the best arm. The explore-first and epsilon-greedy bounds grow with the two-thirds power of the number of rounds, while the UCB1 bound grows only with about the square root, so UCB1 has the strongest guarantee of the three.

When an agent's actions also change the state of its environment, we model both the rewards and the transitions between states using a Markov decision process.

___